In [1]:
import os
import sys
import random
import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
from torchvision import datasets
from torch.utils.data import Dataset, DataLoader

seed = 42
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Using device: cuda


In [3]:
import shutil

# Step 1: Find your dataset root
input_root = "/kaggle/input"

# (Optional) print structure to locate your dataset
for dirname, _, filenames in os.walk(input_root):
    print(dirname)

print("Files Loaded")

#actual dataset path after inspecting above
dataset_path = "/kaggle/input/datasets/fahadullaha/facial-emotion-recognition-dataset/processed_data"


# Step 2: Create output folders
output_path = "/kaggle/working/dataset_new"

os.makedirs(os.path.join(output_path, "negative"), exist_ok=True)
os.makedirs(os.path.join(output_path, "neutral"), exist_ok=True)
os.makedirs(os.path.join(output_path, "positive"), exist_ok=True)

# Step 3: Define class groups
negative_classes = ["angry", "disgust", "fear", "sad"]
neutral_classes = ["neutral"]
positive_classes = ["happy"]

# Step 4: Copy function
def copy_images(class_list, target_folder):
    for cls in class_list:
        class_path = os.path.join(dataset_path, cls)
        
        if not os.path.exists(class_path):
            print(f"Skipping missing folder: {cls}")
            continue
        
        for file in os.listdir(class_path):
            src = os.path.join(class_path, file)
            dst = os.path.join(output_path, target_folder, f"{cls}_{file}")
            
            shutil.copy(src, dst)

# Step 5: Run it
copy_images(negative_classes, "negative")
copy_images(neutral_classes, "neutral")
copy_images(positive_classes, "positive")

print("Data Relabeled")

/kaggle/input
/kaggle/input/datasets
/kaggle/input/datasets/fahadullaha
/kaggle/input/datasets/fahadullaha/facial-emotion-recognition-dataset
/kaggle/input/datasets/fahadullaha/facial-emotion-recognition-dataset/processed_data
/kaggle/input/datasets/fahadullaha/facial-emotion-recognition-dataset/processed_data/surprise
/kaggle/input/datasets/fahadullaha/facial-emotion-recognition-dataset/processed_data/fear
/kaggle/input/datasets/fahadullaha/facial-emotion-recognition-dataset/processed_data/angry
/kaggle/input/datasets/fahadullaha/facial-emotion-recognition-dataset/processed_data/neutral
/kaggle/input/datasets/fahadullaha/facial-emotion-recognition-dataset/processed_data/sad
/kaggle/input/datasets/fahadullaha/facial-emotion-recognition-dataset/processed_data/disgust
/kaggle/input/datasets/fahadullaha/facial-emotion-recognition-dataset/processed_data/happy
Files Loaded
Data Relabeled


In [4]:
#Check if images were loaded successfully
base = "/kaggle/working/dataset_new"
for cls in os.listdir(base):
    print(cls, len(os.listdir(os.path.join(base, cls))))

neutral 8166
positive 11398
negative 24295


In [5]:
from sklearn.model_selection import train_test_split

base = "/kaggle/working/dataset_new"
train_dir = "/kaggle/working/train"
val_dir = "/kaggle/working/val"

os.makedirs(train_dir, exist_ok=True)
os.makedirs(val_dir, exist_ok=True)

for cls in os.listdir(base):
    files = os.listdir(os.path.join(base, cls))
    
    train_files, val_files = train_test_split(files, test_size=0.2, random_state=42)
    
    os.makedirs(os.path.join(train_dir, cls), exist_ok=True)
    os.makedirs(os.path.join(val_dir, cls), exist_ok=True)
    
    for f in train_files:
        shutil.copy(os.path.join(base, cls, f),
                    os.path.join(train_dir, cls, f))
        
    for f in val_files:
        shutil.copy(os.path.join(base, cls, f),
                    os.path.join(val_dir, cls, f))

In [6]:
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor()
])

train_data = datasets.ImageFolder("/kaggle/working/train", transform=transform)
val_data = datasets.ImageFolder("/kaggle/working/val", transform=transform)

train_loader = DataLoader(train_data, batch_size=32, shuffle=True, num_workers = 2)
val_loader = DataLoader(val_data, batch_size=32, num_workers = 2)

In [8]:
class Net(nn.Module):
    def __init__(self, activation):
        super().__init__()

        self.activation = activation

        self.conv1 = nn.Conv2d(3, 16, 3, padding=1)
        self.conv2 = nn.Conv2d(16, 32, 3, padding=1)

        self.pool = nn.MaxPool2d(2, 2)
        self.adaptive_pool = nn.AdaptiveAvgPool2d((7, 7))

        self.fc1 = nn.Linear(32 * 7 * 7, 128)
        self.fc2 = nn.Linear(128, 3)

    def forward(self, x):
        x = self.pool(self.activation(self.conv1(x)))
        x = self.pool(self.activation(self.conv2(x)))

        x = self.adaptive_pool(x)

        x = torch.flatten(x, 1)

        x = self.activation(self.fc1(x))
        x = self.fc2(x)

        return x

In [15]:
# activation functions to test
activations = {
    "ReLU" : nn.ReLU(),
    "Sigmoid" : nn.Sigmoid()
}

In [ ]:
epochs = 5
results = {}

for func_name, activation in activations.items():
    print(f"===Training with {func_name}===")

    net = Net(activation).to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(net.parameters(), lr=0.001)

    train_losses = []
    test_accuracies = []

    for epoch in range(epochs):
        # Train
        net.train()
        running_loss, num_batches = 0.0, 0

        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = net(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            running_loss += loss.item()
            num_batches += 1

        avg_loss = running_loss / num_batches
        train_losses.append(avg_loss)

        # Validate
        net.eval()
        correct, total = 0, 0
        with torch.no_grad():
            for images, labels in val_loader:
                images, labels = images.to(device), labels.to(device)
                
                outputs = net(images)
                _, predicted = torch.max(outputs, 1)
                total += labels.size(0)
                correct += (predicted == labels).sum().item()

        test_acc = 100 * correct / total
        test_accuracies.append(test_acc)
        print(f"Epoch {epoch + 1:02d}/{epochs}, Loss: {avg_loss:.4f}, Test Accuracy: {test_acc:.2f}%")

    results[func_name] = {"train_losses": train_losses, "test_accuracies": test_accuracies}

===Training with ReLU===
Epoch 01/5, Loss: 0.8488, Test Accuracy: 65.03%
Epoch 02/5, Loss: 0.7178, Test Accuracy: 69.26%
